# VneuroTK 工作流：视觉模型特征提取
> `VisionModel` 统一封装 `transformers`、`timm`、`thingsvision` 三种 backend，
> 提供一致的层选取、批量特征提取与结果访问接口。

**工作流**：
1. 准备图像输入
2. 查看本机缓存模型
3. 加载模型并探查层结构
4. 选取目标层（`set_selector`）
5. 批量提取特征（`model.extract()`）
6. 访问与索引特征（`VisualRepresentations`）

In [1]:
import os

import numpy as np
import torch  # type: ignore

import vneurotk as vtk

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # 可选：国内镜像加速

## 1. 准备图像输入

`VisionModel.extract()` 接受 `dict` 批量输入——键为刺激 ID，值为图片。

| 支持的图片格式 | 说明 |
|---|---|
| `PIL.Image` | 直接传入 PIL 图像对象 |
| `np.ndarray` | HWC uint8 数组 |
| `str` / `Path` | 本地图片路径，自动读取 |

> **注意**：不接受 `list`——请使用 `dict` 并为每张图片分配刺激 ID，
> 以确保结果能与 `BaseData.trial_stim_ids` 对齐。

In [2]:
rng = np.random.default_rng(0)
imgs: dict[str, np.ndarray] = {f"img_{i:03d}": rng.integers(0, 255, (224, 224, 3), dtype=np.uint8) for i in range(20)}
print(f"共 {len(imgs)} 张图片，shape: {next(iter(imgs.values())).shape}")

共 20 张图片，shape: (224, 224, 3)


## 2. 查看本机缓存模型

扫描 HuggingFace / timm 默认缓存目录，列出已下载的模型。

In [3]:
vtk.print_cached_models()

  model_id                                               size     last used  
  facebook/dinov2-base                                 330 MB    2026-05-05  
  facebook/dinov3-vitb16-pretrain-lvd1689m             327 MB    2026-05-05  
  facebook/dinov3-vits16-pretrain-lvd1689m              82 MB    2026-05-05  
  google/siglip-base-patch16-224                       778 MB    2026-05-05  
  google/siglip2-base-patch16-224                      1.4 GB    2026-05-05  
  microsoft/resnet-50                                   98 MB    2026-05-05  
  openai/clip-vit-base-patch32                         1.7 GB    2026-05-05  
  Qwen/Qwen3-Embedding-8B                             14.1 GB    2026-05-06  
  timm/resnet50.a1_in1k                                 98 MB    2026-05-05  
  timm/resnetv2_50.a1h_in1k                             98 MB    2026-05-05  
  timm/vit_base_patch16_224.augreg2_in21k_ft_in1k      330 MB    2026-05-05  
  timm/vit_base_patch16_224.dino                       327 MB    2026-05-05  
  ViT-B-32.pt                                          338 MB    2026-05-05

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

██ transformers   ██ timm   ██ clip

## 3. 加载模型

`VisionModel(model_id, backend, device)` 加载指定模型并应用默认层选取策略
（`BlockLevelSelector`，自动识别主干 block）。

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = vtk.VisionModel(
    "facebook/dinov2-base",
    backend="transformers",  # 也可指定 "timm" 或 "thingsvision"
    device=device,
)
print(f"已加载: {model.model_id}  |  默认选取 {len(model.module_names)} 个 module")

Loading transformers model: facebook/dinov2-base (pretrained=True)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded transformers model: facebook/dinov2-base


VisionModel ready | model=facebook/dinov2-base | backend=transformers | modules=12


已加载: facebook/dinov2-base  |  默认选取 12 个 module


查看完整层结构（树状图，含参数量）：

In [5]:
model.print_modules(max_depth=4)

 Module  (⊡ container   • leaf)                              Type                             Param 
 ⊡ ├─ embeddings                                             Dinov2Embeddings       ░░░░░░░░  1.51M 
 ⊡ │  ├─ patch_embeddings                                    Dinov2PatchEmbeddings  ░░░░░░░░ 452.4K 
 • │  │  └─ projection  weight:[768, 3, 14, 14]  bias:[768]  Conv2d                 ░░░░░░░░ 452.4K 
 • │  └─ dropout                                             Dropout                ░░░░░░░░      - 
 ⊡ ├─ encoder                                                Dinov2Encoder          ████████ 85.07M 
 ⊡ │  └─ layer                                               ModuleList             ████████ 85.07M 
 ⊡ │     ├─ 0                                                Dinov2Layer            ░░░░░░░░  7.09M 
 • │     │  ├─ norm1  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ attention                                     Dinov2Attention        ░░░░░░░░  2.36M 
 • │     │  ├─ layer_scale1  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 • │     │  ├─ drop_path                                     Identity               ░░░░░░░░      - 
 • │     │  ├─ norm2  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ mlp                                           Dinov2MLP              ░░░░░░░░  4.72M 
 • │     │  └─ layer_scale2  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 ⊡ │     ├─ 1                                                Dinov2Layer            ░░░░░░░░  7.09M 
 • │     │  ├─ norm1  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ attention                                     Dinov2Attention        ░░░░░░░░  2.36M 
 • │     │  ├─ layer_scale1  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 • │     │  ├─ drop_path                                     Identity               ░░░░░░░░      - 
 • │     │  ├─ norm2  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ mlp                                           Dinov2MLP              ░░░░░░░░  4.72M 
 • │     │  └─ layer_scale2  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 ⊡ │     ├─ 2                                                Dinov2Layer            ░░░░░░░░  7.09M 
 • │     │  ├─ norm1  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ attention                                     Dinov2Attention        ░░░░░░░░  2.36M 
 • │     │  ├─ layer_scale1  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 • │     │  ├─ drop_path                                     Identity               ░░░░░░░░      - 
 • │     │  ├─ norm2  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ mlp                                           Dinov2MLP              ░░░░░░░░  4.72M 
 • │     │  └─ layer_scale2  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 ⊡ │     ├─ 3                                                Dinov2Layer            ░░░░░░░░  7.09M 
 • │     │  ├─ norm1  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ attention                                     Dinov2Attention        ░░░░░░░░  2.36M 
 • │     │  ├─ layer_scale1  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 • │     │  ├─ drop_path                                     Identity               ░░░░░░░░      - 
 • │     │  ├─ norm2  weight:[768]  bias:[768]               LayerNorm              ░░░░░░░░   1.5K 
 ⊡ │     │  ├─ mlp                                           Dinov2MLP              ░░░░░░░░  4.72M 
 • │     │  └─ layer_scale2  lambda1:[768]                   Dinov2LayerScale       ░░░░░░░░    768 
 ⊡ │     ├─ 4                                                

Type legend:

██ Dinov2Embeddings   ██ Dinov2PatchEmbeddings   ██ Conv2d   ██ Dropout   ██ Dinov2Encoder   ██ ModuleList   ██ 
Dinov2Layer   ██ LayerNorm   ██ Dinov2Attention   ██ Dinov2LayerScale   ██ Identity   ██ Dinov2MLP

## 4. 选取目标层（`set_selector`）

提供三种等价方式，可按需组合：

| 方式 | 示例 | 适用场景 |
|---|---|---|
| 按 `module_type` | `set_selector(module_type="Dinov2Layer")` | 一键选同类型 |
| 按 `module_name` | `set_selector(module_name="layernorm")` | 精确指定 |
| 传 `list` | `set_selector([m1, m2, ...])` | 自定义组合 |
| 传 Selector 对象 | `set_selector(BlockLevelSelector())` | 复用预设策略 |

In [6]:
# 方式一：按类型（最简）
model.set_selector(module_type="Dinov2Layer")
print(f"Dinov2Layer: {len(model.module_names)} 层")

Selector updated | modules=12


Dinov2Layer: 12 层


In [7]:
# 方式二：在类型基础上追加特定层名
all_modules = model.module_list
last_norm = [m for m in all_modules if m.module_type == "LayerNorm"][-1]

model.set_selector([m for m in all_modules if m.module_type == "Dinov2Layer"] + [last_norm])
print(f"Dinov2Layer + 最后 LayerNorm: {len(model.module_names)} 层")
print("目标层:", model.module_names)

Selector updated | modules=13


Dinov2Layer + 最后 LayerNorm: 13 层
目标层: ['encoder.layer.0', 'encoder.layer.1', 'encoder.layer.2', 'encoder.layer.3', 'encoder.layer.4', 'encoder.layer.5', 'encoder.layer.6', 'encoder.layer.7', 'encoder.layer.8', 'encoder.layer.9', 'encoder.layer.10', 'encoder.layer.11', 'layernorm']


In [8]:
# 方式三：type + name 联合过滤（取并集）
model.set_selector(module_type="Dinov2Layer", module_name="layernorm")
print(f"联合过滤后: {len(model.module_names)} 层")

Selector updated | modules=13


联合过滤后: 13 层


## 5. 批量特征提取

`model.extract(images, batch_size)` 对 `dict` 图片批量提取，
返回 `VisualRepresentations`（各层激活的有序容器）。

In [9]:
vrs = model.extract(imgs, batch_size=8)
vrs

VisionModel:   0%|          | 0/3 [00:00<?, ?batch/s]

Extracted | n=20 | batch_size=8 | modules=13


VisualRepresentations(20 stimuli x 13 modules, models=['facebook/dinov2-base'])

查看每个 `(model, module)` 组合的元信息：

In [10]:
vrs.meta

,model,module_type,module_name,shape
0,facebook/dinov2-base,Dinov2Layer,encoder.layer.0,"(20, 257, 768)"
1,facebook/dinov2-base,Dinov2Layer,encoder.layer.1,"(20, 257, 768)"
2,facebook/dinov2-base,Dinov2Layer,encoder.layer.2,"(20, 257, 768)"
3,facebook/dinov2-base,Dinov2Layer,encoder.layer.3,"(20, 257, 768)"
4,facebook/dinov2-base,Dinov2Layer,encoder.layer.4,"(20, 257, 768)"
5,facebook/dinov2-base,Dinov2Layer,encoder.layer.5,"(20, 257, 768)"
6,facebook/dinov2-base,Dinov2Layer,encoder.layer.6,"(20, 257, 768)"
7,facebook/dinov2-base,Dinov2Layer,encoder.layer.7,"(20, 257, 768)"
8,facebook/dinov2-base,Dinov2Layer,encoder.layer.8,"(20, 257, 768)"
9,facebook/dinov2-base,Dinov2Layer,encoder.layer.9,"(20, 257, 768)"


## 6. 访问与索引特征

`VisualRepresentations` 支持多种索引方式：

| 索引 | 返回类型 | 说明 |
|---|---|---|
| `vrs["layer_name"]` | `VisualRepresentation` | 按层名取原子对象 |
| `vrs[int]` | `VisualRepresentation` | 按位置取原子对象 |
| `vrs[bool_mask]`（1 个匹配）| `VisualRepresentation` | bool 过滤，单结果 |
| `vrs[bool_mask]`（多个匹配）| `VisualRepresentations` | bool 过滤，子集 |

In [11]:
# 按层名取最后一层
layer_name = model.module_names[-1]
vr = vrs[layer_name]
print("str index → VisualRepresentation")
print(f"  n_stim={vr.n_stim}, shape={vr.shape}")

# 取出 ndarray
arr = vrs.numpy(layer_name)
print(f"\nnumpy()   → ndarray  shape={arr.shape}")

# 转 PyTorch Tensor
t = vrs.to_tensor(layer_name)
print(f"to_tensor → Tensor   shape={t.shape}")

str index → VisualRepresentation
  n_stim=20, shape=(20, 257, 768)

numpy()   → ndarray  shape=(20, 257, 768)
to_tensor → Tensor   shape=torch.Size([20, 257, 768])


In [12]:
# bool mask 过滤：多匹配 → VisualRepresentations 子集
dinov2_mask = vrs.meta["module_type"] == "Dinov2Layer"
subset = vrs[dinov2_mask]
print(f"\nbool mask (多匹配) → {len(subset)} 个 Dinov2Layer")
subset


bool mask (多匹配) → 12 个 Dinov2Layer


VisualRepresentations(20 stimuli x 12 modules, models=['facebook/dinov2-base'])

In [13]:
# bool mask 过滤：单匹配 → 仍返回 VisualRepresentations，通过 [0] 取出 VisualRepresentation
norm_mask = vrs.meta["module_name"] == "layernorm"
vr_norm = vrs[norm_mask]
print(f"\nbool mask (单匹配) → VisualRepresentations，共 {len(vr_norm)} 层")
vr_single = vr_norm[0]
print(f"vr_norm[0] → shape={vr_single.shape}")


bool mask (单匹配) → VisualRepresentations，共 1 层
vr_norm[0] → shape=(20, 257, 768)


### 按刺激 ID 选取子集

`select(ids)` 在所有层上同步对齐，返回新的 `VisualRepresentations`：

In [14]:
subset_ids = list(vrs.stim_ids[:5])
vrs_5 = vrs.select(subset_ids)
print(f"select(前 5 张) → n_stim={vrs_5.n_stim}, stim_ids={vrs_5.stim_ids}")

select(前 5 张) → n_stim=5, stim_ids=('img_000', 'img_001', 'img_002', 'img_003', 'img_004')


## 附：timm 模型示例

换用 ResNet50（timm backend），选取各阶段 Bottleneck 与全局池化层：

In [15]:
model_resnet = vtk.VisionModel(
    "timm/resnet50.a1_in1k",
    backend="timm",
    device=device,
)

# 每阶段最后一个 block + 全局池化（神经科学常用配置）
model_resnet.set_selector(module_name=["layer1.2", "layer2.3", "layer3.5", "layer4.2", "global_pool"])
print(f"ResNet50 目标层: {model_resnet.module_names}")

Loading timm model: timm/resnet50.a1_in1k (pretrained=True)


Loaded timm model: timm/resnet50.a1_in1k


VisionModel ready | model=timm/resnet50.a1_in1k | backend=timm | modules=16


Selector updated | modules=5


ResNet50 目标层: ['layer1.2', 'layer2.3', 'layer3.5', 'layer4.2', 'global_pool']


In [16]:
vrs_resnet = model_resnet.extract(imgs, batch_size=8)
vrs_resnet.meta

VisionModel:   0%|          | 0/3 [00:00<?, ?batch/s]

Extracted | n=20 | batch_size=8 | modules=5


,model,module_type,module_name,shape
0,timm/resnet50.a1_in1k,Bottleneck,layer1.2,"(20, 256, 56, 56)"
1,timm/resnet50.a1_in1k,Bottleneck,layer2.3,"(20, 512, 28, 28)"
2,timm/resnet50.a1_in1k,Bottleneck,layer3.5,"(20, 1024, 14, 14)"
3,timm/resnet50.a1_in1k,Bottleneck,layer4.2,"(20, 2048, 7, 7)"
4,timm/resnet50.a1_in1k,SelectAdaptivePool2d,global_pool,"(20, 2048)"
